In [1]:
import pandas as pd

In [2]:
folder_path = "/mnt/d/all_data/20260821"

In [3]:
file_path=f"{folder_path}/2025_26 Pending Students 161891 Students.xlsx"
sheet_name = "161891 Overall"

In [7]:
df=pd.read_excel(file_path,sheet_name=sheet_name,dtype=str)

df =df[["aadhaar_no","application_id","installment1_amount",
        "district_name","student_name","gender",
        "community","scheme_amount"]]

In [5]:
df.head(10)

,aadhaar_no,application_id,installment1_amount
0,987144883058,TNISSP2026026500232113334380,1310
1,861451512815,TNISSP2026026500232113706197,1310
2,609971418829,TNISSP2026026500232113640611,1310
3,725828842157,TNISSP2026026500232113706994,1310
4,851560063442,TNISSP2026026500232113334211,1310
5,615169879933,TNISSP2026026500232113334387,1310
6,499851249861,TNISSP2026026500232112957232,1310
7,551128944621,TNISSP2026029800265115667876,NaN
8,467643563960,TNISSP2025026500232103539155,2503
9,848997468420,TNISSP2025026500232102983375,1000


In [8]:
df=df.fillna("")
outputpath= '/mnt/d/all_data/20260821/paymenttobe.csv'
df.to_csv(outputpath,index=False)

In [2]:
import pandas as pd

input_file = "/mnt/d/all_data/20260822/Payment pending- 25-26.xlsx"
output_file = "/mnt/d/all_data/20260822/bcmbcpayment.csv"

df = pd.read_excel(input_file)

df.to_csv(output_file, index=False)

print(f"CSV file created: {output_file}")

CSV file created: /mnt/d/all_data/20260822/bcmbcpayment.csv


In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("readCsvfile").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/21 08:48:07 WARN Utils: Your hostname, 2640L, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/21 08:48:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/21 08:48:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/21 08:48:10 WARN Utils: Service 'SparkUI' could

In [26]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, trim


filepath = "/mnt/d/all_data/20260821/payment_date.csv"

# Read multiline CSV
df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv(filepath)
)

# Clean permanent_address
df = df.withColumn(
    "permanent_address",
    trim(
        regexp_replace(
            regexp_replace(
                col("permanent_address"),
                r"[\r\n]+",
                " "
            ),
            r"\s+",
            " "
        )
    )
)

df.show(10, truncate=False)


+----------------------------+----------+----------------------+------+---------+----------+-----------------------------------------------------------------------------+---------------+-----------+----------------+------------+-----------------+
|application_id              |umis_no   |student_name          |gender|community|mobile_no |permanent_address                                                            |districtlgdcode|district_id|district_name   |aadhaar_no  |scholarshipamount|
+----------------------------+----------+----------------------+------+---------+----------+-----------------------------------------------------------------------------+---------------+-----------+----------------+------------+-----------------+
|TNISSP2026043000397106148796|9993122007|DHARSHINI A           |Female|BC       |8870634192|24/16 3rd STREETKOTTAIARANTHANGI TK                                          |588            |29         |Theni           |803445440331|10450            |
|TNISSP20250

In [30]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
df = df.withColumn("aadhaar_no",F.regexp_replace(F.col("aadhaar_no"), r"\.0$", ""))\
        .withColumn("districtlgdcode",F.regexp_replace(F.col("districtlgdcode"), r"\.0$", ""))\
        .withColumn("scholarshipamount",F.col("scholarshipamount").cast(T.DecimalType(18, 0)))

In [31]:
df.show(10, truncate=False)

+----------------------------+----------+----------------------+------+---------+----------+-----------------------------------------------------------------------------+---------------+-----------+----------------+------------+-----------------+
|application_id              |umis_no   |student_name          |gender|community|mobile_no |permanent_address                                                            |districtlgdcode|district_id|district_name   |aadhaar_no  |scholarshipamount|
+----------------------------+----------+----------------------+------+---------+----------+-----------------------------------------------------------------------------+---------------+-----------+----------------+------------+-----------------+
|TNISSP2026043000397106148796|9993122007|DHARSHINI A           |Female|BC       |8870634192|24/16 3rd STREETKOTTAIARANTHANGI TK                                          |588            |29         |Theni           |803445440331|10450            |
|TNISSP20250

In [32]:
# PySpark DataFrame -> Pandas
pdf = df.toPandas()
pdf = pdf.fillna("").astype(str)
# Save Pandas DataFrame as CSV
output_file = "/mnt/d/all_data/20260821/paymentbcmbc.csv"

pdf.to_csv(
    output_file,
    index=False
)

print(f"CSV saved: {output_file}")

/home/yogavarman/venvs/global_env/lib/python3.14/site-packages/pyspark/sql/pandas/conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


CSV saved: /mnt/d/all_data/20260821/paymentbcmbc.csv
